# 02 - Cohérence des données BAAC de 2019 à 2024

Le premier notebook nous a permis de comprendre les quatre tables de 2024 et leurs
relations. Nous élargissons maintenant les contrôles aux six années du projet.

**La question de ce notebook : peut-on réunir ces années dans une même table d'analyse,
et quelles précautions faut-il prendre avant de le faire ?**

Nous allons vérifier les fichiers disponibles, comparer leurs colonnes, contrôler les
identifiants et examiner les valeurs qui demandent un traitement particulier. Chaque
contrôle doit déboucher sur une conséquence concrète pour la préparation.

À la fin, nous aurons un bilan des transformations nécessaires et des points à décider.
La table finale sera construite dans le notebook 3. Les fichiers bruts restent inchangés.


## Périmètre et repères de lecture

Le projet porte sur les **usagers impliqués dans les accidents corporels enregistrés
en France métropolitaine, Corse comprise, de 2019 à 2024**. La future table contiendra
une ligne par usager du fichier source.

Dans cet audit, nous conservons toute la couverture géographique des CSV afin de
contrôler les sources avant de les filtrer. **Les effectifs présentés ici ne sont donc
pas les effectifs de la métropole.** Le filtre sera appliqué pendant la préparation.

La cible retenue sera `1` pour `grav` égal à 2 ou 3 et `0` pour `grav` égal à 1 ou 4.
Une gravité inconnue sera comptabilisée séparément et exclue de l'apprentissage et de
l'évaluation supervisés. Les quatre modalités d'origine resteront disponibles pour
l'analyse descriptive. L'hospitalisation de plus de 24 heures est un indicateur imparfait
de la gravité médicale : nous conserverons cette limite dans l'interprétation.

Sources : les fichiers annuels [BAAC sur data.gouv.fr](https://www.data.gouv.fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024)
et le document ONISR *Description des bases de données annuelles des accidents corporels
de la circulation routière, années 2005 à 2024*, daté du **21 octobre 2025**, fourni dans
le projet sous le nom `docs/references/description-des-bases-de-donnees-annuelles.pdf`.

Une cohérence de structure ne prouve ni l'exhaustivité des accidents enregistrés ni
l'exactitude de chaque renseignement. C'est le niveau de contrôle que nous cherchons ici.


## 1. Vérifier les sources disponibles

Nous attendons quatre fichiers pour chacune des six années, soit 24 CSV. Les identifiants
sont lus comme du texte : ils servent à relier des enregistrements et ne sont pas des
quantités. Les autres types sont inférés par pandas ; ils seront examinés ensuite.


In [31]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.max_colwidth", 100)


In [32]:
# Fonctionne depuis la racine du projet ou le dossier notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "data" / "raw").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Dossier data/raw introuvable. Placez ce notebook dans notebooks/ "
        "et les CSV dans data/raw/2019/ ... data/raw/2024/."
    )
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TABLE_NAMES = ["caracteristiques", "lieux", "vehicules", "usagers"]


In [33]:
def load_baac_table(year, table_name):
    path = RAW_DIR / str(year) / f"{table_name}.csv"
    # Les identifiants et codes géographiques sont des étiquettes, pas des mesures.
    # Lire en texte préserve notamment les zéros initiaux et les codes 2A/2B.
    header = pd.read_csv(path, sep=";", encoding="utf-8-sig", nrows=0)
    identifier_cols = [
        "Num_Acc", "Accident_Id", "id_vehicule", "id_usager", "num_veh", "dep", "com"
    ]
    dtypes = {col: "string" for col in identifier_cols if col in header.columns}
    return pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=dtypes, low_memory=False)


In [34]:
YEARS = list(range(2019, 2025))
inventory = pd.DataFrame([
    {"année": year, "table": name,
     "présent": (RAW_DIR / str(year) / f"{name}.csv").is_file()}
    for year in YEARS for name in TABLE_NAMES
])
display(inventory.pivot(index="année", columns="table", values="présent"))
if not inventory["présent"].all():
    missing = inventory.loc[~inventory["présent"], ["année", "table"]]
    raise FileNotFoundError(f"Fichiers manquants :\n{missing.to_string(index=False)}")

raw_tables = {
    year: {name: load_baac_table(year, name) for name in TABLE_NAMES}
    for year in YEARS
}
dimensions = pd.DataFrame([
    {"année": year, "table": name, "lignes": len(df), "colonnes": df.shape[1]}
    for year, tables in raw_tables.items() for name, df in tables.items()
])
display(dimensions.pivot(index="année", columns="table", values="lignes"))
display(dimensions.pivot(index="année", columns="table", values="colonnes"))


table,caracteristiques,lieux,usagers,vehicules
année,,,,
2019,True,True,True,True
2020,True,True,True,True
2021,True,True,True,True
2022,True,True,True,True
2023,True,True,True,True
2024,True,True,True,True


table,caracteristiques,lieux,usagers,vehicules
année,,,,
2019,58840,58840,132977,100710
2020,47744,47744,105295,81066
2021,56518,56518,129248,97315
2022,55302,55302,126662,94493
2023,54822,70860,125789,93585
2024,54402,70248,125187,92678


table,caracteristiques,lieux,usagers,vehicules
année,,,,
2019,15,18,15,11
2020,15,18,15,11
2021,15,18,16,11
2022,15,18,16,11
2023,15,18,16,11
2024,15,18,16,11


**Ce que nous observons.** Les 24 fichiers sont présents. Les volumes varient d'une année
à l'autre, ce qui ne constitue pas en soi une anomalie de données. Deux différences
méritent cependant un contrôle : `usagers` passe de 15 à 16 colonnes en 2021, et le nombre
de lignes de `lieux` dépasse celui de `caracteristiques` à partir de 2023.

Nous allons expliquer ces écarts avant d'envisager une concaténation.


## 2. Comparer les colonnes et les types

### 2.1 Les différences de schéma

Nous prenons 2024 comme référence de comparaison. Cela permet de repérer les écarts de
nommage ; cela ne signifie pas que le schéma le plus récent est automatiquement correct.
Les tableaux ci-dessous n'affichent que les différences.


In [35]:
schema_rows = []
for year in YEARS:
    for name in TABLE_NAMES:
        columns = set(raw_tables[year][name].columns)
        reference = set(raw_tables[2024][name].columns)
        if columns != reference:
            schema_rows.append({
                "année": year, "table": name,
                "absentes_par_rapport_à_2024": ", ".join(sorted(reference - columns)),
                "supplémentaires_par_rapport_à_2024": ", ".join(sorted(columns - reference)),
            })
schema_differences = pd.DataFrame(schema_rows)
display(schema_differences)


,année,table,absentes_par_rapport_à_2024,supplémentaires_par_rapport_à_2024
0,2019,usagers,id_usager,
1,2020,usagers,id_usager,
2,2022,caracteristiques,Num_Acc,Accident_Id


En 2022, l'identifiant de l'accident s'appelle `Accident_Id` dans `caracteristiques`,
contre `Num_Acc` dans les autres années. Nous utilisons le même nom dans des copies en
mémoire pour poursuivre les contrôles. La couverture des identifiants sera vérifiée en section 3.

`id_usager` n'existe pas en 2019 et 2020. Il faudra conserver cette différence : nous ne
pouvons pas reconstituer un identifiant individuel officiel à partir du seul véhicule.
Un véhicule peut transporter plusieurs personnes.

Les identifiants de véhicule et d'usager contiennent aussi des espaces. Nous les retirons
uniquement dans les copies d'audit, en vérifiant que cette opération ne fusionne pas des
identifiants auparavant distincts. Le renommage et cette normalisation ne constituent pas
encore la préparation de la table finale.


In [36]:
audit_tables = {}
normalization_rows = []
identifier_cols = ["Num_Acc", "id_vehicule", "id_usager", "num_veh", "dep", "com"]
for year in YEARS:
    audit_tables[year] = {}
    for name, source in raw_tables[year].items():
        df = source.rename(columns={"Accident_Id": "Num_Acc"}).copy()
        for col in identifier_cols:
            if col not in df:
                continue
            before = df[col].astype("string")
            after = before.str.replace(r"\s+", "", regex=True).replace("", pd.NA)
            changed = before.fillna("<NA>").ne(after.fillna("<NA>"))
            # Plusieurs formes brutes d'un même identifiant normalisé signalent une collision.
            pairs = pd.DataFrame({"avant": before, "après": after}).dropna().drop_duplicates()
            collisions = pairs.groupby("après")["avant"].nunique().gt(1).sum()
            normalization_rows.append({
                "année": year, "table": name, "variable": col,
                "valeurs_modifiées": int(changed.sum()),
                "nouvelles_valeurs_manquantes": int((before.notna() & after.isna()).sum()),
                "identifiants_avec_collision": int(collisions),
            })
            df[col] = after
        audit_tables[year][name] = df
normalization_report = pd.DataFrame(normalization_rows)
display(normalization_report.loc[
    normalization_report[["valeurs_modifiées", "nouvelles_valeurs_manquantes",
                          "identifiants_avec_collision"]].ne(0).any(axis=1)
])


,année,table,variable,valeurs_modifiées,nouvelles_valeurs_manquantes,identifiants_avec_collision
5,2019,vehicules,id_vehicule,100710,0,0
8,2019,usagers,id_vehicule,132977,0,0
15,2020,vehicules,id_vehicule,81066,0,0
18,2020,usagers,id_vehicule,105295,0,0
25,2021,vehicules,id_vehicule,97315,0,0
28,2021,usagers,id_vehicule,129248,0,0
29,2021,usagers,id_usager,129248,0,0
36,2022,vehicules,id_vehicule,94493,0,0
39,2022,usagers,id_vehicule,126662,0,0
40,2022,usagers,id_usager,126662,0,0


Les espaces concernent notamment les identifiants de véhicule sur les six années et
les identifiants d'usager depuis 2021. Aucun rapprochement entre plusieurs formes brutes
distinctes n'est détecté après normalisation. Cette opération sera donc reprise de façon
explicite dans la préparation, sur toutes les tables concernées.

### 2.2 Un même champ n'est pas toujours lu avec le même type

Les types ci-dessous sont ceux obtenus à la lecture, après avoir imposé le texte pour les
identifiants. Nous affichons les variables dont le type varie ou qui sont absentes certaines
années. Un type numérique ne suffit pas à reconnaître une mesure : beaucoup de nombres
dans BAAC sont des codes de catégories.


In [37]:
type_rows = [
    {"année": year, "table": name, "variable": col, "type_lu": str(df[col].dtype)}
    for year, tables in audit_tables.items() for name, df in tables.items()
    for col in df.columns
]
type_comparison = pd.DataFrame(type_rows).pivot(
    index=["table", "variable"], columns="année", values="type_lu"
).fillna("ABSENTE")
display(type_comparison.loc[type_comparison.nunique(axis=1).gt(1)])


année                 2019     2020     2021     2022     2023     2024
table   variable                                                       
lieux   larrout    float64      str      str      str      str      str
        lartpc     float64      str      str      str      str      str
        nbv          int64    int64    int64      str      str      str
        v1         float64    int64    int64    int64    int64    int64
usagers an_nais      int64    int64  float64  float64  float64  float64
        id_usager  ABSENTE  ABSENTE   string   string   string   string

`an_nais`, par exemple, est lu comme un entier en 2019-2020, puis comme un nombre décimal
dans les fichiers qui contiennent des valeurs manquantes. Des champs de `lieux`, dont
`nbv`, `larrout` et `lartpc`, changent également de type lu.

**Conséquence pour la préparation :** nous définirons les types d'après le sens des
variables et leur contenu. Les années de naissance pourront utiliser un entier acceptant
les valeurs manquantes. Les conversions des mesures devront conserver la trace des
valeurs qui ne se convertissent pas. Nous ne convertirons pas tous les codes en nombres :
`actp`, par exemple, possède aussi des modalités `A` et `B`.


## 3. Vérifier les identifiants et les relations

### 3.1 Complétude, unicité et lignes identiques

Nous attendons une clé d'accident unique dans `caracteristiques` et une clé de véhicule
unique sur le couple `(Num_Acc, id_vehicule)`. Pour les usagers, nous pouvons contrôler
`id_usager` à partir de 2021 seulement.

Nous comptons aussi les lignes entièrement identiques. Le nombre correspond aux répétitions
au-delà de la première ligne. Dans `usagers`, une ligne identique n'est pas forcément une
personne enregistrée deux fois : deux personnes peuvent partager tous les attributs
disponibles, surtout en l'absence d'identifiant individuel.


In [38]:
key_rows, duplicate_rows = [], []
for year, tables in audit_tables.items():
    keys = {"caracteristiques": ["Num_Acc"],
            "vehicules": ["Num_Acc", "id_vehicule"]}
    if "id_usager" in tables["usagers"]:
        keys["usagers"] = ["id_usager"]
    for name, key in keys.items():
        df = tables[name]
        key_rows.append({
            "année": year, "table": name, "clé": " + ".join(key),
            "lignes_clé_incomplète": int(df[key].isna().any(axis=1).sum()),
            "lignes_clé_répétée": int(df.duplicated(key, keep=False).sum()),
        })
    for name, df in tables.items():
        duplicate_rows.append({"année": year, "table": name,
                               "répétitions_exactes": int(df.duplicated().sum())})
key_report = pd.DataFrame(key_rows)
duplicate_report = pd.DataFrame(duplicate_rows)
display(key_report)
display(duplicate_report.pivot(index="année", columns="table", values="répétitions_exactes"))


,année,table,clé,lignes_clé_incomplète,lignes_clé_répétée
0,2019,caracteristiques,Num_Acc,0,0
1,2019,vehicules,Num_Acc + id_vehicule,0,0
2,2020,caracteristiques,Num_Acc,0,0
3,2020,vehicules,Num_Acc + id_vehicule,0,0
4,2021,caracteristiques,Num_Acc,0,0
5,2021,vehicules,Num_Acc + id_vehicule,0,0
6,2021,usagers,id_usager,0,0
7,2022,caracteristiques,Num_Acc,0,0
8,2022,vehicules,Num_Acc + id_vehicule,0,0
9,2022,usagers,id_usager,0,0


table,caracteristiques,lieux,usagers,vehicules
année,,,,
2019,0,0,101,0
2020,0,0,63,0
2021,0,0,0,0
2022,0,0,0,0
2023,0,0,0,0
2024,0,2,0,0


Les clés testées sont complètes et uniques. Dans `usagers`, nous trouvons néanmoins
101 répétitions exactes en 2019 et 63 en 2020. **Nous conservons ces lignes** : sans
identifiant individuel, les supprimer ferait courir le risque de retirer de vrais usagers.
En 2024, `lieux` contient aussi deux répétitions exactes, à examiner avec sa multiplicité.

### 3.2 Les tables décrivent-elles les mêmes accidents et véhicules ?

Le contrôle des accidents compare les ensembles d'identifiants dans les quatre fichiers.
Nous vérifions ensuite que chaque usager retrouve un véhicule sur la clé composée et que
la jointure conserve son nombre de lignes. `validate="many_to_one"` rend le contrôle
explicite : plusieurs usagers peuvent partager un véhicule, mais pas l'inverse dans cette
jointure. Ce résultat ne suffit pas, à lui seul, à valider toutes les autres jointures.


In [39]:
coverage_rows, link_rows = [], []
for year, tables in audit_tables.items():
    accident_ids = set(tables["caracteristiques"]["Num_Acc"].dropna())
    for name, df in tables.items():
        ids = set(df["Num_Acc"].dropna())
        coverage_rows.append({
            "année": year, "table": name,
            "Num_Acc_manquants": int(df["Num_Acc"].isna().sum()),
            "accidents_absents_des_caractéristiques": len(ids - accident_ids),
            "accidents_des_caractéristiques_absents": len(accident_ids - ids),
        })
    users, vehicles = tables["usagers"], tables["vehicules"]
    vehicle_key = ["Num_Acc", "id_vehicule"]
    linked = users.merge(vehicles[vehicle_key + ["num_veh"]], on=vehicle_key,
                         how="left", validate="many_to_one", indicator=True,
                         suffixes=("_usager", "_véhicule"))
    vehicles_linked = vehicles[vehicle_key].merge(
        users[vehicle_key].drop_duplicates(), on=vehicle_key,
        how="left", validate="one_to_one", indicator=True)
    mismatch = (linked["num_veh_usager"].notna() & linked["num_veh_véhicule"].notna()
                & linked["num_veh_usager"].ne(linked["num_veh_véhicule"]))
    link_rows.append({
        "année": year,
        "usagers_clé_véhicule_incomplète": int(users[vehicle_key].isna().any(axis=1).sum()),
        "usagers_sans_véhicule": int(linked["_merge"].eq("left_only").sum()),
        "num_veh_contradictoires": int(mismatch.sum()),
        "lignes_usagers_conservées": len(linked) == len(users),
        "véhicules_sans_usager": int(vehicles_linked["_merge"].eq("left_only").sum()),
    })
coverage_report = pd.DataFrame(coverage_rows)
link_report = pd.DataFrame(link_rows)
display(coverage_report.groupby("année").sum(numeric_only=True))
display(link_report)


,Num_Acc_manquants,accidents_absents_des_caractéristiques,accidents_des_caractéristiques_absents
année,,,
2019,0,0,0
2020,0,0,0
2021,0,0,0
2022,0,0,0
2023,0,0,0
2024,0,0,0


,année,usagers_clé_véhicule_incomplète,usagers_sans_véhicule,num_veh_contradictoires,lignes_usagers_conservées,véhicules_sans_usager
0,2019,0,0,0,True,2040
1,2020,0,0,0,True,1899
2,2021,0,0,0,True,6
3,2022,0,0,0,True,0
4,2023,0,0,0,True,40
5,2024,0,0,0,True,24


Les ensembles d'accidents concordent dans les quatre tables, pour chaque année. Tous les
usagers retrouvent leur véhicule et la jointure conserve leurs effectifs.

En revanche, certains véhicules n'ont pas d'usager associé : 2 040 en 2019, 1 899 en 2020,
6 en 2021, aucun en 2022, 40 en 2023 et 24 en 2024. Le PDF précise que les 24 cas de 2024
correspondent à des conducteurs en fuite non créés dans le fichier des usagers. Nous ne
généralisons pas cette explication à tous les cas des autres années sans vérification.

Pour notre unité d'analyse, **la table `usagers` servira de point de départ**. Nous
n'inventerons pas de personnes pour compléter les véhicules sans usager. Cette différence
de couverture devra être gardée en tête lors des comparaisons annuelles.


### 3.3 Le cas particulier de la table `lieux`

Le nombre de lignes par accident change à partir de 2023. Nous mesurons sa fréquence et
regardons si des variables utiles à l'analyse diffèrent entre les lignes d'un même accident.
Les valeurs manquantes sont comptées comme une modalité distincte pour ce contrôle :
une valeur renseignée et une valeur manquante constituent aussi une différence à examiner.


In [40]:
location_rows, location_conflicts = [], []
location_fields = ["catr", "circ", "nbv", "prof", "plan", "surf", "infra", "situ", "vma"]
for year, tables in audit_tables.items():
    locations = tables["lieux"]
    counts = locations.groupby("Num_Acc").size()
    multiple_ids = counts.index[counts.gt(1)]
    location_rows.append({
        "année": year, "accidents": len(counts),
        "accidents_avec_plusieurs_lieux": len(multiple_ids),
        "part_en_%": round(100 * len(multiple_ids) / len(counts), 2),
        "maximum_de_lignes": int(counts.max()),
    })
    if len(multiple_ids):
        variation = (locations.loc[locations["Num_Acc"].isin(multiple_ids)]
                     .groupby("Num_Acc")[location_fields].nunique(dropna=False).gt(1).sum())
        location_conflicts.append({"année": year, **variation.to_dict()})
location_report = pd.DataFrame(location_rows)
display(location_report)
display(pd.DataFrame(location_conflicts).set_index("année"))


,année,accidents,accidents_avec_plusieurs_lieux,part_en_%,maximum_de_lignes
0,2019,58840,0,0.00,1
1,2020,47744,0,0.00,1
2,2021,56518,0,0.00,1
3,2022,55302,0,0.00,1
4,2023,54822,15870,28.95,5
5,2024,54402,15645,28.76,5


,catr,circ,nbv,prof,plan,surf,infra,situ,vma
année,,,,,,,,,
2023,0,6018,8570,2808,2362,350,2352,1368,5169
2024,0,5781,8158,2756,2397,188,2370,1401,4711


In [41]:
# Un exemple concret : une jointure directe doublerait les lignes des usagers.
example_year, example_accident = 2024, "202400000005"
example_locations = audit_tables[example_year]["lieux"].query("Num_Acc == @example_accident")
example_users = audit_tables[example_year]["usagers"].query("Num_Acc == @example_accident")
display(example_locations[["Num_Acc", "voie", *location_fields]])
display(pd.Series({
    "usagers_avant_jointure": len(example_users),
    "lignes_lieux": len(example_locations),
    "lignes_après_jointure_directe": len(example_users.merge(example_locations, on="Num_Acc")),
}, name="effet_sur_un_accident"))


,Num_Acc,voie,catr,circ,nbv,prof,plan,surf,infra,situ,vma
6,202400000005,BEDOULE (CHEMIN DE LA),1,2,4,1,1,2,0,1,50
7,202400000005,BEDOULE (CHEMIN DE LA),1,2,4,2,1,2,0,1,70


usagers_avant_jointure           4
lignes_lieux                     2
lignes_après_jointure_directe    8
Name: effet_sur_un_accident, dtype: int64

En 2019-2022, chaque accident possède une ligne dans `lieux`. En 2023 et 2024, près de
29 % des accidents en possèdent plusieurs, avec un maximum de cinq lignes.

Ces lignes ne sont pas toutes interchangeables : `vma` diffère pour 5 169 accidents en
2023 et 4 711 en 2024. Dans l'exemple affiché, quatre usagers deviennent huit lignes après
une jointure directe, et les deux lignes de lieu indiquent des vitesses maximales différentes.

La réponse **DSR ONISR du 12 décembre 2024**, dans la discussion « lieux-2023 » de
[data.gouv.fr](https://www.data.gouv.fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024/discussions),
explique que plusieurs lieux sont décrits pour les accidents à une intersection. Cela
n'explique pas à lui seul pourquoi cette représentation apparaît dans nos exports en 2023.
Les CSV n'exposent pas de clé permettant de rattacher chaque ligne de lieu à un véhicule.

**À l'issue de cet audit, la représentation des lieux doit être définie.** Prendre arbitrairement la première ligne
ou la vitesse maximale ferait perdre une information sans justification. Il faudra choisir,
variable par variable, les informations conservées et une représentation des désaccords.
`catr` est stable entre les lignes multiples des fichiers étudiés ; ce constat ne règle pas
le cas des autres variables. La table de lieux préparée devra finalement présenter une
seule ligne par accident avant de rejoindre les usagers. Le notebook 03 détaille la
solution retenue : valeur connue unique, désaccord et information partielle, variable par variable.


## 4. Examiner les valeurs qui demandent un traitement

### 4.1 Comparer les codes observés au dictionnaire

Nous transcrivons ci-dessous les listes de codes du PDF pour les variables catégorielles
contrôlées. `place`, décrit par un schéma de placement, n'est pas inclus. Les identifiants,
mesures et textes libres ne relèvent pas de cette comparaison.

Un code absent de cette liste est **un écart à examiner**, pas une autorisation de supprimer
la ligne. La documentation concerne plusieurs millésimes et les pratiques de saisie peuvent
évoluer. Nous séparons aussi les valeurs réellement manquantes des codes spéciaux.


In [42]:
DOCUMENTED_CODES = {
    "caracteristiques": {
        "lum": [1, 2, 3, 4, 5], "agg": [1, 2], "int": list(range(1, 10)),
        "atm": [-1, *range(1, 10)], "col": [-1, *range(1, 8)],
    },
    "lieux": {
        "catr": [1, 2, 3, 4, 5, 6, 7, 9], "circ": [-1, 1, 2, 3, 4],
        "vosp": [-1, 0, 1, 2, 3], "prof": [-1, 1, 2, 3, 4],
        "plan": [-1, 1, 2, 3, 4], "surf": [-1, *range(1, 10)],
        "infra": [-1, *range(0, 10)], "situ": [-1, 0, 1, 2, 3, 4, 5, 6, 8],
    },
    "vehicules": {
        "senc": [-1, 0, 1, 2, 3], "catv": [*range(0, 22), *range(30, 44), 50, 60, 80, 99],
        "obs": [-1, *range(0, 18)], "obsm": [-1, 0, 1, 2, 4, 5, 6, 9],
        "choc": [-1, *range(0, 10)], "manv": [-1, *range(0, 27)],
        "motor": [-1, *range(0, 7)],
    },
    "usagers": {
        "catu": [1, 2, 3], "grav": [1, 2, 3, 4], "sexe": [1, 2],
        "trajet": [-1, 0, 1, 2, 3, 4, 5, 9],
        "secu1": [-1, *range(0, 10)], "secu2": [-1, *range(0, 10)],
        "secu3": [-1, *range(0, 10)], "locp": [-1, *range(0, 10)],
        "actp": [-1, 0, 1, 2, 3, 4, 5, 6, 9, "A", "B"], "etatp": [-1, 1, 2, 3],
    },
}

def canonical_codes(series):
    """Comparer 1, 1.0 et '1' sans perdre les modalités alphabétiques."""
    return (series.astype("string").str.strip().replace({"": pd.NA, ".": pd.NA})
            .str.replace(r"^(-?\d+)\.0+$", r"\1", regex=True))

unexpected_rows, completeness_rows = [], []
for year, tables in audit_tables.items():
    for name, variables in DOCUMENTED_CODES.items():
        for col, allowed in variables.items():
            values = canonical_codes(tables[name][col])
            allowed_text = {str(value) for value in allowed}
            unexpected = values.notna() & ~values.isin(allowed_text)
            for value, count in values[unexpected].value_counts().items():
                unexpected_rows.append({
                    "année": year, "table": name, "variable": col, "code_observé": value,
                    "lignes": int(count), "part_en_%": round(100 * count / len(values), 4),
                })
            completeness_rows.append({
                "année": year, "table": name, "variable": col,
                "manquants_ou_vides": int(values.isna().sum()),
                "code_-1": int(values.eq("-1").sum()), "code_0": int(values.eq("0").sum()),
            })
unexpected_codes = pd.DataFrame(unexpected_rows)
completeness_report = pd.DataFrame(completeness_rows)
display(unexpected_codes)


,année,table,variable,code_observé,lignes,part_en_%
0,2019,usagers,actp,7,18,0.0135
1,2019,usagers,actp,8,13,0.0098
2,2020,caracteristiques,lum,-1,5,0.0105
3,2020,caracteristiques,int,-1,1,0.0021
4,2020,vehicules,catv,-1,5,0.0062
5,2020,usagers,actp,7,11,0.0104
6,2020,usagers,actp,8,8,0.0076
7,2021,vehicules,catv,-1,4,0.0041
8,2021,usagers,grav,-1,60,0.0464
9,2021,usagers,sexe,-1,3062,2.3691


Les écarts ne se limitent pas à des valeurs rares et isolées. `sexe = -1` apparaît depuis
2021 ; `grav = -1` est présent entre 2021 et 2023. Nous observons aussi `actp = 7` ou `8`,
absents de la liste du PDF fourni, ainsi que quelques `-1` dans d'autres variables.
Ces valeurs seront conservées dans les sources et traitées explicitement dans les données
préparées. Nous ne leur attribuons pas de nouvelle signification sans justification.

Certaines catégories de `catv` figurent dans le dictionnaire mais y sont indiquées comme
inutilisées depuis 2006. Nous vérifions aussi ce cas plutôt que de considérer leur seule
présence dans le dictionnaire comme suffisante.


In [43]:
RETIRED_CATV = {4, 5, 6, 8, 9, 11, 12, 18, 19}
display(pd.DataFrame([
    {"année": year, "véhicules_avec_ancien_code_catv": int(
        canonical_codes(audit_tables[year]["vehicules"]["catv"])
        .isin({str(value) for value in RETIRED_CATV}).sum())}
    for year in YEARS
]))

# Extrait volontairement limité aux variables qui illustrent les différents cas.
selected_missing = completeness_report.loc[
    (completeness_report["table"].eq("usagers")
     & completeness_report["variable"].isin(["grav", "sexe", "trajet", "secu1"]))
    | (completeness_report["table"].eq("lieux")
       & completeness_report["variable"].isin(["infra", "vosp"]))
]
display(selected_missing.pivot(index=["table", "variable"], columns="année",
                               values=["manquants_ou_vides", "code_-1", "code_0"]))


,année,véhicules_avec_ancien_code_catv
0,2019,0
1,2020,0
2,2021,0
3,2022,0
4,2023,0
5,2024,0


manquants_ou_vides                          code_-1       \
année                          2019 2020 2021 2022 2023 2024    2019 2020   
table   variable                                                            
lieux   infra                     0    0    0    0    0    0      45  878   
        vosp                      0    0    0    0    0    0     669  279   
usagers grav                      0    0    0    0    0    0       0    0   
        secu1                     0    0    0    0    0    0      88   33   
        sexe                      0    0    0    0    0    0       0    0   
        trajet                    0    0    0    0    0    0     543  213   

                                         code_0                              \
année             2021  2022  2023  2024   2019   2020   2021   2022   2023   
table   variable                                                              
lieux   infra      619   694   923   812  49041  39539  47504  46386  59086   
        vosp       210   164  3954  3832  53719  42623  49789  48248  57756   
usagers grav        60   241   118     0      0      0      0      0      0   
        secu1     2791  2679  2292  2103   7196   9962  11880  12144  11938   
        sexe      3062  2744  2431  2395      0      0      0      0      0   
        trajet    3270  2874  2499  2626  34376  26793  32567  32737  33303   

                         
année              2024  
table   variable         
lieux   infra     58188  
        vosp      57329  
usagers grav          0  
        secu1     12216  
        sexe          0  
        trajet    35024

**Nous ne remplacerons pas tous les zéros par des valeurs manquantes.** Leur sens dépend
de la variable : `secu1 = 0` signifie « aucun équipement », `infra = 0` signifie « aucun »
et `vosp = 0` signifie « sans objet ». Pour `trajet`, le zéro correspond à une information
non renseignée. Les `-1` documentés correspondent aussi à des informations non renseignées,
mais tous les champs ne documentent pas cette modalité.

La préparation devra distinguer une absence d'information d'une absence d'équipement ou
d'infrastructure. Nous garderons les modalités d'origine pour pouvoir vérifier ces choix.

### 4.2 La gravité et la future cible

Nous affichons les quatre modalités officielles ainsi que les codes effectivement
observés. Le tableau de cible est un contrôle de comptage ; il ne produit pas encore
le jeu d'apprentissage.


In [44]:
grav_counts = pd.DataFrame({
    year: canonical_codes(audit_tables[year]["usagers"]["grav"])
          .fillna("MANQUANT").value_counts()
    for year in YEARS
}).fillna(0).astype(int)
display(grav_counts.sort_index())

target_rows = []
for year, tables in audit_tables.items():
    grav = canonical_codes(tables["usagers"]["grav"])
    positive = int(grav.isin(["2", "3"]).sum())
    negative = int(grav.isin(["1", "4"]).sum())
    unknown = len(grav) - positive - negative
    target_rows.append({
        "année": year, "cible_1": positive, "cible_0": negative, "gravité_inconnue": unknown,
        "part_cible_1_parmi_connus_%": round(100 * positive / (positive + negative), 2),
    })
target_report = pd.DataFrame(target_rows)
display(target_report)


,2019,2020,2021,2022,2023,2024
grav,,,,,,
-1,0,0,60,241,118,0
1,55314,43267,55143,53630,53399,52920
2,3498,2780,3219,3550,3398,3432
3,20858,16775,19093,19260,19271,19126
4,53307,42473,51733,49981,49603,49709


,année,cible_1,cible_0,gravité_inconnue,part_cible_1_parmi_connus_%
0,2019,24356,108621,0,18.32
1,2020,19555,85740,0,18.57
2,2021,22312,106876,60,17.27
3,2022,22810,103611,241,18.04
4,2023,22669,103002,118,18.04
5,2024,22558,102629,0,18.02


Les fichiers contiennent 419 usagers avec `grav = -1` : 60 en 2021, 241 en 2022 et 118
en 2023. Ils ne doivent être rattachés à aucune des deux classes. Le fichier 2024 ne
présente pas de gravité inconnue : 22 558 usagers appartiennent à la classe 1 et 102 629 à
la classe 0, sur l'ensemble de la couverture géographique du fichier.

Ces effectifs changeront après le filtre métropolitain. La proportion de classe 1 présentée
ici décrit les usagers enregistrés dont la gravité est connue ; elle ne mesure pas la
probabilité d'un accident sur la route.

### 4.3 Dates, coordonnées et années de naissance

Ces contrôles cherchent des incohérences simples : une date impossible, une année différente
du dossier source, une heure non interprétable, des coordonnées hors des limites mondiales,
ou une année de naissance conduisant à un âge suspect.

L'âge calculé ici est une approximation `année de l'accident - année de naissance`.
Le seuil de 110 ans sert à signaler des lignes à examiner ; il ne constitue pas une règle
automatique d'exclusion. De même, des coordonnées situées dans les limites mondiales
peuvent malgré tout être erronées.


In [45]:
sanity_rows, old_birth_rows = [], []
for year, tables in audit_tables.items():
    carac, users = tables["caracteristiques"], tables["usagers"]
    dates = pd.to_datetime(pd.DataFrame({
        "year": pd.to_numeric(carac["an"], errors="coerce"),
        "month": pd.to_numeric(carac["mois"], errors="coerce"),
        "day": pd.to_numeric(carac["jour"], errors="coerce"),
    }), errors="coerce")
    times = pd.to_datetime(carac["hrmn"].astype("string"), format="%H:%M", errors="coerce")
    lat = pd.to_numeric(carac["lat"].astype("string").str.replace(",", ".", regex=False), errors="coerce")
    lon = pd.to_numeric(carac["long"].astype("string").str.replace(",", ".", regex=False), errors="coerce")
    birth = pd.to_numeric(users["an_nais"], errors="coerce")
    age = year - birth
    sanity_rows.append({
        "année": year, "dates_non_interprétables": int(dates.isna().sum()),
        "année_différente_du_dossier": int(dates.dt.year.ne(year).sum()),
        "heures_non_interprétables": int(times.isna().sum()),
        "coordonnées_non_interprétables": int((lat.isna() | lon.isna()).sum()),
        "coordonnées_hors_limites": int((lat.notna() & ~lat.between(-90, 90)
                                        | lon.notna() & ~lon.between(-180, 180)).sum()),
        "coordonnées_0_0": int((lat.eq(0) & lon.eq(0)).sum()),
        "naissances_manquantes_ou_non_interprétables": int(birth.isna().sum()),
        "âges_négatifs": int(age.lt(0).sum()), "âges_supérieurs_à_110": int(age.gt(110).sum()),
    })
    for value, count in birth[age.gt(110)].value_counts().items():
        old_birth_rows.append({"année": year, "année_de_naissance": int(value), "usagers": int(count)})
sanity_report = pd.DataFrame(sanity_rows)
display(sanity_report.set_index("année").T)
display(pd.DataFrame(old_birth_rows))


année,2019,2020,2021,2022,2023,2024
dates_non_interprétables,0,0,0,0,0,0
année_différente_du_dossier,0,0,0,0,0,0
heures_non_interprétables,0,0,0,0,0,0
coordonnées_non_interprétables,0,0,0,0,0,0
coordonnées_hors_limites,0,0,0,0,0,0
coordonnées_0_0,0,0,0,0,0,0
naissances_manquantes_ou_non_interprétables,0,0,3067,2874,2598,2579
âges_négatifs,0,0,0,0,0,0
âges_supérieurs_à_110,176,15,0,0,0,0


,année,année_de_naissance,usagers
0,2019,1900,133
1,2019,1901,43
2,2020,1900,15


Les dates, heures et coordonnées passent ces contrôles élémentaires. Nous ne concluons
pas pour autant que les localisations ont toutes été correctement renseignées.

Les années de naissance demandent un traitement : des valeurs manquent depuis 2021 et
191 usagers de 2019-2020 ont un âge approximatif supérieur à 110 ans. Les années de
naissance concernées sont 1900 et 1901. Elles suggèrent des valeurs de remplacement
possibles, mais cet audit ne permet pas de le démontrer. Nous les signalerons avant de
choisir une règle documentée pour l'âge. Aucune valeur n'est corrigée ici.


### 4.4 Vérifier la présentation des codes géographiques

Le département suffit au filtre métropolitain prévu, mais il ne garantit pas que le
code commune est exploitable. Nous contrôlons son format sur cinq caractères et l'accord
de son préfixe avec le département déclaré. Le contrôle est limité à la métropole et ne
remplace pas une vérification contre le référentiel historique de l'INSEE.


In [46]:
geo_rows = []
METRO_CODES_AUDIT = {f"{i:02d}" for i in range(1, 96) if i != 20} | {"2A", "2B"}
for year, tables in audit_tables.items():
    carac = tables["caracteristiques"]
    dep = carac["dep"].str.upper().str.zfill(2)
    com = carac["com"].str.upper()
    metro = dep.isin(METRO_CODES_AUDIT)
    format_ok = com.str.fullmatch(r"(?:[0-9]{5}|2[AB][0-9]{3})", na=False)
    prefix_ok = com.str[:2].eq(dep).fillna(False)
    geo_rows.append({
        "année": year,
        "accidents_métropole_département": int(metro.sum()),
        "commune_format_à_vérifier": int((metro & ~format_ok).sum()),
        "commune_département_discordants": int((metro & format_ok & ~prefix_ok).sum()),
    })
geo_report = pd.DataFrame(geo_rows)
display(geo_report)


,année,accidents_métropole_département,commune_format_à_vérifier,commune_département_discordants
0,2019,56016,0,3
1,2020,45121,0,0
2,2021,53540,0,0
3,2022,52380,1,0
4,2023,51641,0,0
5,2024,51058,0,0


Nous trouvons trois discordances commune-département en 2019 et un code commune `N/C`
en 2022. Nous ne pouvons pas choisir quelle information corriger dans les discordances
sans vérification supplémentaire. Le notebook 03 conserve ces accidents dans le périmètre
défini par leur département et signale les communes à vérifier. Le code non interprétable
reste accessible dans sa colonne brute.


## 5. Bilan et préparation du notebook 3

**Les six années peuvent être réunies, à condition d'harmoniser les sources et de traiter
la multiplicité de `lieux` avant les jointures.** Les clés et les correspondances entre
tables sont cohérentes dans les fichiers étudiés. Une concaténation suivie de jointures
sans ces précautions ne suffirait cependant pas à construire une table fiable par usager.

| Constat | Années concernées | Suite à donner |
| --- | --- | --- |
| `Accident_Id` remplace `Num_Acc` dans `caracteristiques` | 2022 | Renommer explicitement avant les contrôles et les jointures. |
| Espaces dans les identifiants | 2019-2024 | Normaliser de la même manière dans les tables liées et contrôler les collisions. |
| Absence d'identifiant individuel officiel | 2019-2020 | Préserver chaque ligne source ; prévoir un identifiant technique traçable. |
| Usagers aux attributs entièrement identiques | 2019-2020 | Conserver les lignes, faute de preuve d'un double enregistrement. |
| Véhicules sans usager associé | Plusieurs années | Partir des usagers pour les jointures et documenter cette couverture. |
| Plusieurs lieux par accident, avec des valeurs différentes | 2023-2024 | **Décider la règle de représentation avant de construire les jointures.** |
| Types lus variables selon les années | Plusieurs variables | Fixer les types attendus et tracer les échecs de conversion. |
| Codes spéciaux et codes absents de la liste du PDF | Plusieurs variables | **Définir les traitements par variable**, sans remplacement global des zéros. |
| `grav = -1` | 2021-2023 | Conserver la trace ; exclure ces cibles inconnues des analyses supervisées. |
| Naissances manquantes ou âges suspects | Selon les années | **Décider une règle pour l'âge**, avec indicateur de valeur manquante si nécessaire. |
| Sources couvrant aussi l'outre-mer | 2019-2024 | Filtrer la métropole, Corse comprise, à partir des codes géographiques vérifiés. |

Les décisions à prendre à l'issue de cet audit concernent la représentation des lieux,
les valeurs particulières des variables et le calcul de l'âge. Elles sont maintenant
explicitées et appliquées dans le notebook 03. Cet audit conserve les constats avant
préparation pour permettre la comparaison entre les étapes.

### Comment vérifier que la préparation a réussi ?

Le notebook suivant devra conserver une ligne par usager retenu dans le périmètre,
produire un bilan des effectifs à chaque étape et vérifier que chaque jointure préserve
cette unité. Les exclusions de cibles inconnues seront dénombrées, et les transformations
devront pouvoir être reliées aux valeurs sources.

La séparation temporelle reste celle du cahier des charges : apprentissage sur 2019-2022,
validation sur 2023 et test final sur 2024. Ce notebook et le précédent ont déjà examiné
les données de 2024 ; cette exploration préalable sera déclarée. Le test final ne servira
pas à choisir les variables, les modèles ou leurs réglages à partir de leurs performances.

